In [50]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

In [51]:
df_books = pd.read_csv('books.csv')

In [52]:
df_ratings = pd.read_csv('ratings.csv')

In [53]:
df_full = pd.merge(df_ratings, df_books, on='book_id')

In [54]:
df_full = df_full.drop(columns=['image_url', 'small_image_url','ratings_1','ratings_2','ratings_3','ratings_4','ratings_5', 'isbn', 'isbn13', 'work_id', 'best_book_id', 'original_publication_year', 'books_count', 'title', 'language_code', 'work_ratings_count', 'work_text_reviews_count'], errors='ignore')

In [55]:
m = df_full['ratings_count'].quantile(0.90)
C = df_full['average_rating'].mean()

def weighted_rating(row, m=m, C=C):
    v = row['ratings_count']
    R = row['average_rating']
    return round((v / (v + m)) * R + (m / (v + m)) * C, 2)

df_full['score'] = df_full.apply(weighted_rating, axis=1)

In [56]:
df_full = df_full.drop(columns=['average_rating', 'ratings_count'])

In [57]:
df = df_full.sort_values('score', ascending=False)

In [58]:
display(df.head())

,book_id,user_id,rating,id,authors,original_title,score
43,1,23576,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
35,1,18361,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
20,1,10610,5,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
19,1,10335,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
18,1,10246,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45


In [59]:
df.shape

(79701, 7)

In [60]:
df_books['combined_features'] = df_books['original_title'].fillna('') + ' ' + df_books['authors'].fillna('')

In [61]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_books['combined_features'])

cosine_sim_item_based = cosine_similarity(tfidf_matrix)
indices_item_based = df_books.reset_index().drop_duplicates(subset='original_title', keep='first').set_index('original_title')['index']

Item-based

In [62]:
def get_item_based_recommendations(title, num_recommendations=5):
    idx = indices_item_based[title]

    sim_scores = list(enumerate(cosine_sim_item_based[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]

    book_indices = [i[0] for i in sim_scores]

    return df['original_title'].iloc[book_indices]

In [63]:
print("Рекомендации:")
display(get_item_based_recommendations('The Hobbit or There and Back Again'))

Рекомендации:


15249                          Pride and Prejudice
39295                     The Pillars of the Earth
128      Harry Potter and the Order of the Phoenix
10975                         La sombra del viento
1754                         The Lord of the Rings
Name: original_title, dtype: str

User-based

In [64]:
ratings = df[
    [
        'user_id',
        'book_id',
        'rating',
        'original_title'
    ]
].copy()

ratings['user_id'] = ratings['user_id'].astype(str)
ratings['book_id'] = ratings['book_id'].astype(str)

ratings = (
    ratings
    .groupby(
        ['user_id', 'book_id'],
        as_index=False
    )
    .agg({
        'rating': 'mean',
        'original_title': 'first'
    })
)

user_counts = ratings['user_id'].value_counts()
active_users = user_counts[
    user_counts >= 10
].index
ratings_filtered = ratings[
    ratings['user_id'].isin(active_users)
].copy()

book_counts = ratings_filtered['book_id'].value_counts()
popular_books = book_counts[
    book_counts >= 5
].index
ratings_filtered = ratings_filtered[
    ratings_filtered['book_id'].isin(popular_books)
].copy()

user_ids = ratings_filtered['user_id'].unique()
book_ids = ratings_filtered['book_id'].unique()

user_to_idx = {
    user_id: i
    for i, user_id in enumerate(user_ids)
}

book_to_idx = {
    book_id: i
    for i, book_id in enumerate(book_ids)
}

idx_to_book = {
    idx: book_id
    for book_id, idx in book_to_idx.items()
}

rows = ratings_filtered['user_id'].map(user_to_idx)
cols = ratings_filtered['book_id'].map(book_to_idx)
values = ratings_filtered['rating'].values

user_item_matrix = csr_matrix(
    (
        values,
        (rows.values, cols.values)
    ),
    shape=(
        len(user_ids),
        len(book_ids)
    )
)

book_titles = (
    ratings_filtered
    .drop_duplicates('book_id')
    .set_index('book_id')['original_title']
    .to_dict()
)

def get_similar_users(
    user_id,
    n_neighbors=10
):
    user_id = str(user_id)
    if user_id not in user_to_idx:
        return pd.DataFrame(
            columns=[
                'user_id',
                'similarity'
            ]
        )
    user_idx = user_to_idx[user_id]
    similarities = cosine_similarity(
        user_item_matrix[user_idx],
        user_item_matrix
    ).flatten()
    similar_indices = similarities.argsort()[::-1]
    similar_indices = similar_indices[
        similar_indices != user_idx
    ]
    similar_indices = similar_indices[
        :n_neighbors
    ]
    result = []
    for idx in similar_indices:
        result.append({
            'user_id': user_ids[idx],
            'similarity': similarities[idx]
        })
    return pd.DataFrame(result)

def get_user_based_recommendations(
    user_id,
    num_recommendations=5,
    n_neighbors=10
):
    user_id = str(user_id)
    if user_id not in user_to_idx:
        return pd.DataFrame(
            columns=[
                'book_id',
                'original_title',
                'score'
            ]
        )
    user_idx = user_to_idx[user_id]
    similarities = cosine_similarity(
        user_item_matrix[user_idx],
        user_item_matrix
    ).flatten()
    similar_indices = similarities.argsort()[::-1]
    similar_indices = similar_indices[
        similar_indices != user_idx
    ]
    similar_indices = similar_indices[
        :n_neighbors
    ]
    user_books = set(
        user_item_matrix[user_idx].indices
    )
    weighted_scores = {}
    similarity_sums = {}
    for similar_idx in similar_indices:
        similarity = similarities[similar_idx]
        if similarity <= 0:
            continue
        row = user_item_matrix[similar_idx]
        book_indices = row.indices
        ratings_values = row.data
        for book_idx, rating in zip(
            book_indices,
            ratings_values
        ):
            if book_idx in user_books:
                continue
            if rating >= 4:
                if book_idx not in weighted_scores:
                    weighted_scores[book_idx] = 0
                    similarity_sums[book_idx] = 0
                weighted_scores[book_idx] += (
                    similarity * rating
                )
                similarity_sums[book_idx] += (
                    similarity
                )
    recommendation_scores = {}
    for book_idx in weighted_scores:
        if similarity_sums[book_idx] > 0:
            recommendation_scores[book_idx] = (
                weighted_scores[book_idx]
                / similarity_sums[book_idx]
            )
    recommendations = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    recommendations = recommendations[
        :num_recommendations
    ]
    result = []
    for book_idx, score in recommendations:
        book_id = idx_to_book[book_idx]
        title = book_titles.get(
            book_id,
            'Unknown'
        )
        result.append({
            'book_id': book_id,
            'original_title': title,
            'score': round(score, 4)
        })
    return pd.DataFrame(result)

In [69]:
user_id = '22154'
print("ВЫБРАННЫЙ ПОЛЬЗОВАТЕЛЬ:", user_id)

print("\nПохожие пользователи:")
similar_users = get_similar_users(
    user_id,
    n_neighbors=10
)
print(similar_users)

print("\nРекомендации:")
recommendations = get_user_based_recommendations(
    user_id,
    num_recommendations=5,
    n_neighbors=10
)
print(recommendations)

ВЫБРАННЫЙ ПОЛЬЗОВАТЕЛЬ: 22154

Похожие пользователи:
  user_id  similarity
0   10008    0.607052
1    1350    0.456979
2   32139    0.426636
3   12417    0.426014
4   38067    0.413297
5   10120    0.386444
6   51618    0.329292
7    7994    0.313986
8   38865    0.312031
9   25155    0.309552

Рекомендации:
  book_id                                     original_title  score
0     360                                    Mostly Harmless    5.0
1    1362                                           Ἰστορίαι    5.0
2     357                 The Long Dark Tea-Time of the Soul    5.0
3    1869   Nickel and Dimed: On (Not) Getting By in America    5.0
4    2767  A People's History of the United States: 1492 ...    5.0


Cold start

In [66]:
book_stats = (
    df.groupby(
        ['book_id', 'original_title'],
        as_index=False
    )
    .agg(
        rating_mean=('rating', 'mean'),
        rating_count=('rating', 'count')
    )
)

global_mean = book_stats['rating_mean'].mean()

MIN_RATINGS = 20

popular_books = book_stats[
    book_stats['rating_count'] >= MIN_RATINGS
].copy()

m = MIN_RATINGS
C = global_mean

popular_books['cold_start_score'] = (
    (
        popular_books['rating_count']
        /
        (
            popular_books['rating_count'] + m
        )
    )
    *
    popular_books['rating_mean']

    +

    (
        m
        /
        (
            popular_books['rating_count'] + m
        )
    )
    *
    C
)

popular_books = popular_books.sort_values(
    'cold_start_score',
    ascending=False
)

def get_cold_start_recommendations(
    num_recommendations=5
):

    recommendations = popular_books[
        [
            'book_id',
            'original_title',
            'rating_mean',
            'rating_count',
            'cold_start_score'
        ]
    ].head(num_recommendations).copy()


    recommendations = recommendations.rename(
        columns={
            'rating_mean': 'average_rating',
            'rating_count': 'number_of_ratings',
            'cold_start_score': 'score'
        }
    )


    recommendations['score'] = (
        recommendations['score']
        .round(4)
    )


    return recommendations.reset_index(
        drop=True
    )

cold_start_recommendations = (
    get_cold_start_recommendations(5)
)

print("\nCold start рекомендации:")
print(cold_start_recommendations)


Cold start рекомендации:
   book_id                                     original_title  average_rating  \
0     9566                         Still Life with Woodpecker        4.777778   
1     4708                           The Beautiful and Damned        4.660000   
2     9569                                    Villa Incognito        4.618557   
3     3885                         The Taste of Home Cookbook        4.550000   
4     2767  A People's History of the United States: 1492 ...        4.540000   

   number_of_ratings   score  
0                 99  4.6243  
1                100  4.5275  
2                 97  4.4897  
3                100  4.4358  
4                100  4.4275  
